In [1]:
from mmpose.datasets import builder
from mmengine.runner import Runner
from mmengine import Config

cfg = Config.fromfile(
	r'C:\Users\user\Documents\GitHub\mmpose\my_code\custom_config\HMD_xregopose_imple_config_1backbone.py'
	# r'C:\Users\user\Documents\GitHub\mmpose\my_code\custom_config\HMD_mo2cap2_imple_config_2backbone.py'
	# r'C:\Users\user\Documents\GitHub\mmpose\my_code\custom_config\HMD_mo2cap2_imple_config.py'
	# r'C:\Users\user\Documents\GitHub\mmpose\my_code\custom_config\HMD_mo2cap2_config.py'
    # r'C:\Users\user\Documents\GitHub\mmpose\temp_modify\custom_config\HMD_coco_config.py' # img 에서 3d pose로 나올 수 있도록 해야 함
	# r'C:\Users\user\Documents\GitHub\mmpose\temp_modify\custom_config\HMD_h36m_config.py' # pose lift : 2d pose 가 인풋
	# r'C:\Users\user\Documents\GitHub\mmpose\configs\body_3d_keypoint\image_pose_lift\h36m\image-pose-lift_tcn_8xb64-200e_h36m.py'
)
cfg.work_dir = 'work_dirs/HMD_mo2cap2_test'
cfg.randomness = dict(seed=42)
runner = Runner.from_cfg(cfg)

11/26 14:33:51 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: win32
    Python: 3.8.13 (default, Oct 19 2022, 22:38:03) [MSC v.1916 64 bit (AMD64)]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 42
    GPU 0: NVIDIA GeForce RTX 3090
    CUDA_HOME: C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v11.6
    NVCC: Cuda compilation tools, release 11.6, V11.6.112
    MSVC: n/a, reason: fileno
    PyTorch: 1.12.1+cu116
    PyTorch compiling details: PyTorch built with:
  - C++ Version: 199711
  - MSVC 192829337
  - Intel(R) Math Kernel Library Version 2020.0.2 Product Build 20200624 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v2.6.0 (Git Hash 52b5f107dd9cf10910aaa19cb47f3abf9b349815)
  - OpenMP 2019
  - LAPACK is enabled (usually provided by MKL)
  - CPU capability usage: AVX2
  - CUDA Runtime 11.6
  - NVCC architecture flags: -gencode;arch=compute_37,code=sm_37;-gencode;a

In [ ]:
runner.train() 

11/26 14:36:31 - mmengine - INFO - load model from: C:\Users\user\Downloads\pytorch-20240821T053436Z-001\pytorch\pose_coco\coco_pose_resnet_101_256x192.pth.tar
11/26 14:36:31 - mmengine - INFO - Loads checkpoint by local backend from path: C:\Users\user\Downloads\pytorch-20240821T053436Z-001\pytorch\pose_coco\coco_pose_resnet_101_256x192.pth.tar
11/26 14:36:31 - mmengine - WARNING - The model and loaded state dict do not match exactly

unexpected key in source state_dict: deconv_layers.0.weight, deconv_layers.1.weight, deconv_layers.1.bias, deconv_layers.1.running_mean, deconv_layers.1.running_var, deconv_layers.3.weight, deconv_layers.4.weight, deconv_layers.4.bias, deconv_layers.4.running_mean, deconv_layers.4.running_var, deconv_layers.6.weight, deconv_layers.7.weight, deconv_layers.7.bias, deconv_layers.7.running_mean, deconv_layers.7.running_var, final_layer.weight, final_layer.bias

11/26 14:36:31 - mmengine - WARNING - "FileClient" will be deprecated in future. Please use io fun

In [3]:
import gc
gc.collect()
raise

RuntimeError: No active exception to reraise

In [ ]:
def extract_and_combine_numbers(file_path):
    # Split the file path by the directory separator
    parts = file_path.split('\\')

    # Extract the relevant parts
    chunk_part = parts[-3]  # This is 'mo2cap2_chunk_0002'
    file_part = parts[-1]   # This is 'mo2cap2_chunk_0002_000028.png'

    # Extract the numbers and strip leading zeros
    chunk_number = chunk_part.split('_')[-1].lstrip('0')
    file_number = file_part.split('_')[-1].split('.')[0].lstrip('0')

    # Combine the numbers
    combined_number = int(chunk_number + file_number)
    
    return combined_number

# Example usage
file_path = 'F:\\mo2cap2_data_temp_extracted\\TrainSet\\mo2cap2_chunk_0020\\rgba\\mo2cap2_chunk_0002_010728.png'
result = extract_and_combine_numbers(file_path)
print(result)  # Output should be 228

In [ ]:
temp_dataset_from_runner = Runner.build_dataloader(cfg.train_dataloader)

In [ ]:
len(temp_dataset_from_runner)

In [ ]:
for i in temp_dataset_from_runner:
	print(i)

In [ ]:
METAPATH = r'C:\Users\user\Documents\GitHub\mmpose\temp_modify\HMD_metainfo\annotation_definitions.json'
with open(METAPATH,'r') as _meta:
	METAINFO: dict = json.load(_meta)

In [ ]:


class temp_dataset():
	

	ROOT_DIRS = ['rgba','depth','json','objectId']
	def __init__(self, ann_file):
		self.ann_file = ann_file
		self.index = self._load_index()

	def index_db(self):
		return self._index_dir(self.ann_file)
		
	def _load_index(self):
		"""Get indexed set. If the set has already been
		indexed, load the file, otherwise index it and save cache.

		Returns:
			dict -- index set
		"""

		idx_path = os.path.join(self.ann_file, 'index.h5')
		
		if os.path.exists(idx_path):
			return self.read_h5(idx_path)

		index = self.index_db()
		self.write_h5(idx_path, index)
		return index

	def get_files(self, path, formats=None):
		"""Get files contained in path

		Arguments:
			path {str} -- path

		Keyword Arguments:
			formats {str/list} -- file formats; if None take all (default: {None})

		Returns:
			list -- file names
			list -- file paths
		"""

		if formats:

			if isinstance(formats, str):
				formats = [formats]
			else:
				assert isinstance(formats, list)

			names = []
			paths = []

			for f_format in formats:
				files = [f for f in os.listdir(path)
						if re.match(r'.*\.{}'.format(f_format), f)]

				files.sort()
				names.extend(files)
				paths.extend([os.path.join(path, f) for f in files])

			return names, paths

		names = os.listdir(path)
		names.sort()
		paths = [os.path.join(path, f) for f in names]

		return names, paths	

	def write_h5(self, path, data):
		"""Write h5 file

		Arguments:
			path {str} -- file path where to save the data
			data {seriaizable} -- data to be saved

		Raises:
			NotImplementedError -- non serializable data to save
		"""

		if '.h5' not in path[-3:]:
			path += '.h5'

		hf = h5py.File(path, 'w')

		if isinstance(data, dict):
			for k, v in data.items():
				if isinstance(v[0], str):
					v = [a.encode('utf8') for a in v]
				hf.create_dataset(k, data=v)
		elif isinstance(data, list):
			hf.create_dataset('val', data=data)
		elif isinstance(data, np.ndarray):
			hf.create_dataset('val', data=data)
		else:
			raise NotImplementedError
		hf.close()

	def read_h5(self, path):
		"""Load data from h5 file

		Arguments:
			path {str} -- file path

		Raises:
			FileNotFoundError -- Path not pointing to a file

		Returns:
			dict -- dictionary containing the data
		"""

		if not os.path.isfile(path):
			raise FileNotFoundError()

		data_files = dict()
		h5_data = h5py.File(path)
		tags = list(h5_data.keys())
		for tag in tags:
			tag_data = np.asarray(h5_data[tag]).copy()
			data_files.update({tag: tag_data})
		h5_data.close()

		return data_files	


	def get_subdirs(self, path):
		"""Get directories contained in path

		Arguments:
			path {str} -- path

		Returns:
			list -- directory names
			list -- directory paths
		"""

		try:
			names = os.walk(path).next()[1]
		except AttributeError:
			names = next(os.walk(path))[1]

		names.sort()
		dir_paths = [os.path.join(path, n) for n in names]

		return names, dir_paths

	def _index_dir(self, path):
		"""Recursively add paths to the set of
		indexed files

		Arguments:
			path {str} -- folder path

		Returns:
			dict -- indexed files per root dir
		"""

		indexed_paths = dict()
		sub_dirs, _ = self.get_subdirs(path)

		if set(self.ROOT_DIRS) <= set(sub_dirs):

			# get files from subdirs
			n_frames = -1

			# let's extract the rgba and json data per frame
			for sub_dir in self.ROOT_DIRS:
				d_path = os.path.join(path, sub_dir)
				_, paths = self.get_files(d_path)

				if n_frames < 0:
					n_frames = len(paths)
				else:
					if len(paths) != n_frames:
						raise('Frames info in {} not matching other passes'.format(d_path))

				encoded = [p.encode('utf8') for p in paths]
				indexed_paths.update({sub_dir: encoded})

			return indexed_paths

		# initialize indexed_paths
		for sub_dir in self.ROOT_DIRS:
			indexed_paths.update({sub_dir: []})

		# check subdirs of path and merge info
		for sub_dir in sub_dirs:
			indexed = self._index_dir(os.path.join(path, sub_dir))

			for r_dir in self.ROOT_DIRS:
				indexed_paths[r_dir].extend(indexed[r_dir])

		return indexed_paths

In [ ]:
import os
import re
import h5py
import numpy as np

class temp_dataset():
    def __init__(self, ann_file):
        self.ann_file = ann_file
        self.index = self._load_index()
        print(self.index)

    def index_db(self):
        return self._index_dir(self.ann_file)
        
    def _load_index(self):
        idx_path = os.path.join(self.ann_file, 'index.h5')
        
        if os.path.exists(idx_path):
            return self.read_h5(idx_path)

        index = self.index_db()
        self.write_h5(idx_path, index)
        return index

    def write_h5(self, path, data):
        if '.h5' not in path[-3:]:
            path += '.h5'

        hf = h5py.File(path, 'w')

        if isinstance(data, dict):
            for k, v in data.items():
                if isinstance(v[0], str):
                    v = [a.encode('utf8') for a in v]
                hf.create_dataset(k, data=v)
        elif isinstance(data, list):
            hf.create_dataset('val', data=data)
        elif isinstance(data, np.ndarray):
            hf.create_dataset('val', data=data)
        else:
            raise NotImplementedError
        hf.close()

    def read_h5(self, path):
        if not os.path.isfile(path):
            raise FileNotFoundError()

        data_files = dict()
        h5_data = h5py.File(path)
        tags = list(h5_data.keys())
        for tag in tags:
            tag_data = np.asarray(h5_data[tag]).copy()
            data_files.update({tag: tag_data})
        h5_data.close()

        return data_files

    def _index_dir(self, path):
        indexed_paths = {
            'rgba': [],
            'depth': [],
            'frame_data': [],
            'segmentation': [],
            
        }

        for root, dirs, files in os.walk(path):
            if root.split(os.path.sep)[-1].startswith('sequence.'):
                
                for file in files:
                    full_path = os.path.join(root, file)
                    if file.endswith('camera.png'):
                        indexed_paths['rgba'].append(full_path.encode('utf8'))
                    elif file.endswith('Depth.exr'):
                        indexed_paths['depth'].append(full_path.encode('utf8'))
                    elif file.endswith('frame_data.json'):
                        indexed_paths['frame_data'].append(full_path.encode('utf8'))
                    elif file.endswith('instance segmentation.png'):
                        indexed_paths['segmentation'].append(full_path.encode('utf8'))

        return indexed_paths

In [ ]:
data_path = r'C:\Users\user\AppData\LocalLow\DefaultCompany\perception tutorial\solo_59'
temp = temp_dataset(data_path)

In [ ]:
temp.index

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

# Load image
image_path = r'C:\Users\user\AppData\LocalLow\DefaultCompany\perception tutorial\solo_64\sequence.2\step0.camera.png'
image = Image.open(image_path)

# Create figure and axis
fig, ax = plt.subplots()

# Display the image
ax.imshow(image)

# Bounding box parameters
origin = (411.0, 446.0)
dimension = (171.0, 205.0)

# Create a Rectangle patch
rect = patches.Rectangle(origin, dimension[0], dimension[1], linewidth=1, edgecolor='r', facecolor='none')

# Add the patch to the Axes
ax.add_patch(rect)

# Display the plot
plt.show()

In [ ]:
import json
import numpy as np
from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple, Union

def parse_data_info(_rgba, _depth, _segmentation, _frame_data) -> Optional[dict]:
	# JSON 파일 읽기
	try:
		with open(_frame_data, 'r') as f:
			frame_data = json.load(f)
	except FileNotFoundError:
		print(f"Error: File not found - {_frame_data}")
		return None
	except json.JSONDecodeError:
		print(f"Error: Invalid JSON format in file - {_frame_data}")
		return None

	# 키포인트 정보 추출
	keypoints_info = frame_data['captures'][0]['annotations'][0]['values'][0]['keypoints']

	# 3D 키포인트 정보 추출
	keypoint3d_info = frame_data['captures'][0]['annotations'][5]['keypoints'][0]['keypoints']

	# 사용할 joint_name 목록 생성
	valid_joint_names = []
	for j in METAINFO['annotationDefinitions'][2]['template']['keypoints']:
		if j['label']=='nose' : valid_joint_names.append(j['label'])
		elif '_' in j['label'] :
			temp_joint_ = j['label'].split('_')
			valid_joint_names.append(temp_joint_[-1]+'_'+temp_joint_[0])
	assert valid_joint_names 
	# 키포인트 및 가시성 정보 생성
	keypoints = []
	keypoints_visible = []
	keypoint3d = []
	num_keypoints = 0

	for kp in keypoints_info:
		if kp['state'] != 0:
			keypoints.extend(kp['location'])
			keypoints_visible.append(1)
			num_keypoints += 1
		else:
			keypoints.extend([0, 0])
			keypoints_visible.append(0)

	# 3D 키포인트 정보 생성
	for kp in keypoint3d_info:
		if kp['label'] in valid_joint_names:
			keypoint3d.extend(kp['location'])

	keypoints = np.array(keypoints).reshape(1, -1, 2)
	keypoints_visible = np.array(keypoints_visible).reshape(1, -1)
	keypoint3d = np.array(keypoint3d).reshape(1, -1, 3)

	# data_info 딕셔너리 생성
	data_info = {
		'img_id': frame_data['frame'],
		'img_path': _rgba,
		'depth_path': _depth,
		'segmentation_path': _segmentation,
		'num_keypoints': num_keypoints,
		'keypoints': keypoints,
		'keypoints_visible': keypoints_visible,
		'keypoint3d': keypoint3d,
		'raw_ann_info': {
			'rgba': _rgba,
			'depth': _depth,
			'segmentation': _segmentation,
			'frame_data': _frame_data
		}
	}

	return data_info

In [ ]:
_rgba = None
_depth = None
_segmentation = None
_frame_data = r'C:\Users\user\AppData\LocalLow\DefaultCompany\perception tutorial\solo_64\sequence.2\step0.frame_data.json'
# temp_data_info = parse_data_info(_rgba, _depth, _segmentation, _frame_data)

In [ ]:
try:
	with open(_frame_data, 'r') as f:
		frame_data = json.load(f)
except FileNotFoundError:
	print(f"Error: File not found - {_frame_data}")

except json.JSONDecodeError:
	print(f"Error: Invalid JSON format in file - {_frame_data}")


for _cap in frame_data['captures']:
	if isinstance(_cap, dict):
		if isinstance(_cap['annotations'], list):
			keypoints_info = None
			keypoint3d_info = None
			bbox_info = None

			for _ann in _cap['annotations']:
				if isinstance(_ann, dict):
					if _ann.get('@type', '').endswith('KeypointAnnotation'):
						keypoints_info = _ann.get('values', [{}])[0].get('keypoints')
					elif _ann.get('@type', '').endswith('Keypoint3dAnnotation'):
						keypoint3d_info = _ann.get('keypoints', [{}])[0].get('keypoints')
					elif _ann.get('@type', '').endswith('BoundingBox2DAnnotation'):
						bbox_values = _ann.get('values', [{}])[0]
						origin = bbox_values.get('origin', [])
						dimension = bbox_values.get('dimension', [])
						if origin and dimension:
							bbox_info = origin + dimension  # This will give [x, y, w, h]

			# Check if we found all required annotations
			if keypoints_info and keypoint3d_info and bbox_info:
				res = {
					'keypoints_info': keypoints_info,
					'keypoint3d_info': keypoint3d_info,
					'bbox_info': bbox_info,
					# Add other processed data as needed
				}
				print(res)

	# If we've gone through all captures without finding the required data
	print("Error: Could not find all required annotations in any capture")


    # except KeyError as e:
    #     print(f"Error: Missing key in JSON structure - {e}")
    #     return None

In [ ]:
temp_data_info

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import json
%matplotlib widget
# 데이터 로드
data = {
    "keypoints": [
        {
            "instanceId": 21,
            "keypoints": [
                {"label": "hip", "location": [-0.0139369741, 0.5109188, 0.008858096], "orientation": [-0.04940117, -0.03463017, -0.0623554476, 0.996228933]},
                {"label": "leg_left", "location": [-0.07532008, 0.471192718, 0.008057099], "orientation": [-0.3827881, -0.0119570661, -0.08030707, 0.9202615]},
                {"label": "knee_left", "location": [-0.111645691, 0.343151778, 0.14516525], "orientation": [0.438487053, -0.09574184, 0.01873271, 0.8934271]},
                {"label": "accessories_ankle_left", "location": [-0.09675469, 0.229785889, -0.0193558745], "orientation": [0.438487053, -0.09574184, 0.01873271, 0.8934271]},
                {"label": "ankle_left", "location": [-0.09675469, 0.229785889, -0.0193558745], "orientation": [0.5248918, -0.1553334, 0.05377073, 0.8351462]},
                {"label": "foot_left", "location": [-0.106825195, 0.168205172, -0.029754594], "orientation": [0.533636332, -0.154759929, 0.055399593, 0.82958585]},
                {"label": "toes_left", "location": [-0.117891945, 0.1340459, -0.0128959939], "orientation": [0.533636332, -0.154759929, 0.055399593, 0.82958585]},
                {"label": "lower_leg_left", "location": [-0.105546936, 0.297451138, 0.04857389], "orientation": [0.438487053, -0.09574184, 0.01873271, 0.8934271]},
                {"label": "upper_leg_left", "location": [-0.0967326462, 0.399906129, 0.07793829], "orientation": [-0.3827881, -0.0119570661, -0.08030707, 0.9202615]},
                {"label": "leg_right", "location": [0.0354874581, 0.4576292, 0.0164741557], "orientation": [-0.05593412, -0.00219800323, 0.00215756753, 0.9984297]},
                {"label": "knee_right", "location": [0.0428944677, 0.268550158, 0.04278826], "orientation": [0.159043044, -0.0814319551, 0.005150892, 0.983894169]},
                {"label": "accessories_ankle_right", "location": [0.0632615462, 0.08409393, -0.0349783935], "orientation": [0.159043044, -0.0814319551, 0.005150892, 0.983894169]},
                {"label": "ankle_right", "location": [0.0632615462, 0.08409393, -0.0349783935], "orientation": [0.290778726, -0.162592784, -0.06635734, 0.940536261]},
                {"label": "foot_right", "location": [0.04711944, 0.0240537822, -0.004641503], "orientation": [0.1443423, -0.1506083, -0.09031529, 0.9738204]},
                {"label": "toes_right", "location": [0.0403137766, 0.0150246322, 0.0330432728], "orientation": [0.1443423, -0.1506083, -0.09031529, 0.9738204]},
                {"label": "lower_leg_right", "location": [0.0599223971, 0.1766271, -0.009328257], "orientation": [0.159043044, -0.0814319551, 0.005150892, 0.983894169]},
                {"label": "upper_leg_right", "location": [0.0413406864, 0.356265068, 0.0260736458], "orientation": [-0.05593412, -0.00219800323, 0.00215756753, 0.9984297]},
                {"label": "spine_01", "location": [-0.005890387, 0.546428144, -0.0395375974], "orientation": [-0.0408523157, -0.0607394353, -0.0613171235, 0.995430648]},
                {"label": "spine_02", "location": [0.000178989023, 0.607451558, -0.03094288], "orientation": [-0.0538162775, -0.106139645, -0.06494627, 0.9907676]},
                {"label": "spine_03", "location": [0.0106106922, 0.6661722, -0.04563635], "orientation": [-0.0684362, -0.1506352, -0.0703478158, 0.9837058]},
                {"label": "clavicle_left", "location": [0.005863743, 0.774926662, -0.00479026232], "orientation": [-0.061855346, -0.107326493, -0.1868392, 0.974549234]},
                {"label": "shoulder_left", "location": [-0.0922252, 0.79567343, -0.0592408441], "orientation": [0.190347448, -0.09466736, 0.203005478, 0.955821633]},
                {"label": "elbow_left", "location": [-0.107234009, 0.6780517, -0.117463425], "orientation": [-0.108217165, 0.0220216215, 0.165385678, 0.9800267]},
                {"label": "accessories_wrist_left", "location": [-0.150078237, 0.5530617, -0.06760836], "orientation": [-0.108217165, 0.0220216215, 0.165385678, 0.9800267]},
                {"label": "lower_arm_left", "location": [-0.131419241, 0.62015146, -0.0846629143], "orientation": [-0.108217165, 0.0220216215, 0.165385678, 0.9800267]},
                {"label": "wrist_left", "location": [-0.150078237, 0.5530617, -0.06760836], "orientation": [-0.07558897, 0.0353257731, 0.121953219, 0.989023]},
                {"label": "index_01_left", "location": [-0.161969066, 0.498755664, -0.0427246764], "orientation": [-0.05451624, 0.0545333438, 0.293749183, 0.95276767]},
                {"label": "index_02_left", "location": [-0.162450463, 0.4815016, -0.03891255], "orientation": [-0.0465978, 0.0340295061, 0.414891452, 0.9080398]},
                {"label": "index_03_left", "location": [-0.158660263, 0.467655331, -0.03531645], "orientation": [-0.03863239, 0.005451273, 0.493609726, 0.8688084]},
                {"label": "index_04_left", "location": [-0.155769318, 0.456724524, -0.0318791568], "orientation": [-0.03863239, 0.005451273, 0.493609726, 0.8688084]},
                {"label": "middle_01_left", "location": [-0.173711747, 0.499085158, -0.0487046167], "orientation": [-0.0586658269, 0.03465789, 0.170350268, 0.983025134]},
                {"label": "middle_02_left", "location": [-0.177915633, 0.480725467, -0.0454447232], "orientation": [-0.0537702143, 0.0277852975, 0.32656163, 0.9432363]},
                {"label": "middle_03_left", "location": [-0.176649034, 0.4669018, -0.0423591621], "orientation": [-0.0491496138, 0.0110473651, 0.4196828, 0.9062721]},
                {"label": "middle_04_left", "location": [-0.174519211, 0.4540473, -0.0389331], "orientation": [-0.0491496138, 0.0110473651, 0.4196828, 0.9062721]},
                {"label": "palm_left", "location": [-0.171799958, 0.5002116, -0.0619392358], "orientation": [-0.07558897, 0.0353257731, 0.121953219, 0.989023]},
                {"label": "pinky_01_left", "location": [-0.187633991, 0.5084813, -0.0658791959], "orientation": [0.170770764, -0.00116154179, 0.267351568, 0.9483458]},
                {"label": "pinky_02_left", "location": [-0.186134487, 0.498826742, -0.06791073], "orientation": [0.168100864, -0.033163622, 0.428666055, 0.8870669]},
                {"label": "pinky_03_left", "location": [-0.182259649, 0.4893853, -0.07136153], "orientation": [0.162990019, -0.057699427, 0.5409832, 0.8230694]},
                {"label": "pinky_04_left", "location": [-0.176401019, 0.479023933, -0.07364629], "orientation": [0.162990019, -0.057699427, 0.5409832, 0.8230694]},
                {"label": "ring_01_left", "location": [-0.180324435, 0.504873335, -0.0578341], "orientation": [-0.00240878761, 0.0134532116, 0.244485557, 0.969557047]},
                {"label": "ring_02_left", "location": [-0.181567848, 0.486388475, -0.0563926734], "orientation": [-0.00131845847, 0.0054189153, 0.328579068, 0.944460452]},
                {"label": "ring_03_left", "location": [-0.180341452, 0.473127037, -0.0558470376], "orientation": [-0.000287566334, -0.00351977162, 0.3878788, 0.921704054]},
                {"label": "ring_04_left", "location": [-0.179489076, 0.45982632, -0.05422527], "orientation": [-0.000287566334, -0.00351977162, 0.3878788, 0.921704054]},
                {"label": "thumb_01_left", "location": [-0.14514792, 0.529122, -0.0610328428], "orientation": [-0.08873759, 0.1373787, 0.2272985, 0.9599941]},
                {"label": "thumb_02_left", "location": [-0.132969856, 0.510052562, -0.0573838763], "orientation": [-0.102385163, 0.06540503, 0.222732648, 0.967279732]},
                {"label": "thumb_03_left", "location": [-0.130396664, 0.4930307, -0.053724397], "orientation": [-0.121513352, -0.0063560307, 0.216677889, 0.9686306]},
                {"label": "thumb_04_left", "location": [-0.13022542, 0.478723049, -0.05020946], "orientation": [-0.121513352, -0.0063560307, 0.216677889, 0.9686306]},
                {"label": "upper_arm_left", "location": [-0.0962139741, 0.728654861, -0.0826468], "orientation": [0.190347448, -0.09466736, 0.203005478, 0.955821633]},
                {"label": "clavicle_right", "location": [0.01489486, 0.773775458, -0.00183563586], "orientation": [-0.105681792, -0.153349251, 0.0145917069, 0.982396543]},
                {"label": "shoulder_right", "location": [0.128006876, 0.7610588, 0.00439326838], "orientation": [0.161028028, -0.187570229, -0.132727951, 0.959828734]},
                {"label": "elbow_right", "location": [0.171208948, 0.6380463, -0.0167410057], "orientation": [-0.233091891, -0.48463276, -0.281068563, 0.7948587]},
                {"label": "accessories_wrist_right", "location": [0.114836819, 0.5514183, 0.07949172], "orientation": [-0.233091891, -0.48463276, -0.281068563, 0.7948587]},
                {"label": "lower_arm_right", "location": [0.140495211, 0.60252285, 0.03621448], "orientation": [-0.233091891, -0.48463276, -0.281068563, 0.7948587]},
                {"label": "wrist_right", "location": [0.114836819, 0.551418364, 0.07949169], "orientation": [-0.180285409, -0.301991135, -0.2067495, 0.912991762]},
                {"label": "index_01_right", "location": [0.101400077, 0.5107983, 0.115545824], "orientation": [0.205396, 0.265843719, 0.159764215, -0.9282325]},
                {"label": "index_02_right", "location": [0.104525708, 0.4918246, 0.131158859], "orientation": [-0.17609109, -0.260823369, -0.2747526, 0.908556342]},
                {"label": "index_03_right", "location": [0.1028301, 0.481521666, 0.138597742], "orientation": [0.15534322, 0.240524337, 0.351636678, -0.8912739]},
                {"label": "index_04_right", "location": [0.100603573, 0.472160637, 0.145653516], "orientation": [0.15534322, 0.240524337, 0.351636678, -0.8912739]},
                {"label": "middle_01_right", "location": [0.114145033, 0.509539843, 0.115779057], "orientation": [0.122471616, 0.307526439, 0.257958055, -0.9076819]},
                {"label": "middle_02_right", "location": [0.112034909, 0.486604035, 0.126884], "orientation": [-0.0731071755, -0.3074668, -0.3988543, 0.8608341]},
                {"label": "middle_03_right", "location": [0.106614612, 0.47424078, 0.13118799], "orientation": [0.0387304872, 0.29282, 0.4856685, -0.8227292]},
                {"label": "middle_04_right", "location": [0.100014187, 0.463074416, 0.13463971], "orientation": [0.0387304872, 0.29282, 0.4856685, -0.8227292]},
                {"label": "palm_right", "location": [0.119369663, 0.5013942, 0.107383177], "orientation": [-0.180285409, -0.301991135, -0.2067495, 0.912991762]},
                {"label": "pinky_01_right", "location": [0.136583775, 0.510614336, 0.108884543], "orientation": [-0.010371089, 0.2961105, 0.305549234, -0.9049039]},
                {"label": "pinky_02_right", "location": [0.132969618, 0.4973802, 0.110165626], "orientation": [0.0245646164, -0.292238533, -0.347640753, 0.890583932]},
                {"label": "pinky_03_right", "location": [0.13026455, 0.4863147, 0.110705584], "orientation": [-0.0276471153, 0.288556635, 0.3553623, -0.8886448]},
                {"label": "pinky_04_right", "location": [0.127287626, 0.475511938, 0.112398431], "orientation": [-0.0276471153, 0.288556635, 0.3553623, -0.8886448]},
                {"label": "ring_01_right", "location": [0.125873923, 0.508718967, 0.113328651], "orientation": [0.108961955, 0.280916482, 0.18769373, -0.934871733]},
                {"label": "ring_02_right", "location": [0.126640767, 0.489786, 0.122000888], "orientation": [-0.07693575, -0.283287525, -0.294102222, 0.909579]},
                {"label": "ring_03_right", "location": [0.124633737, 0.475722373, 0.1268346], "orientation": [0.0474907234, 0.280913681, 0.385914683, -0.877441049]},
                {"label": "ring_04_right", "location": [0.121558957, 0.465020359, 0.1299221], "orientation": [0.0474907234, 0.280913681, 0.385914683, -0.877441049]},
                {"label": "thumb_01_right", "location": [0.09990986, 0.5324442, 0.08933815], "orientation": [0.1398331, 0.411951, 0.28869158, -0.8528782]},
                {"label": "thumb_02_right", "location": [0.08806057, 0.515641332, 0.08827855], "orientation": [-0.1485774, -0.340072751, -0.296862185, 0.879857063]},
                {"label": "thumb_03_right", "location": [0.08030977, 0.499851, 0.09291394], "orientation": [0.164432436, 0.265874773, 0.304997116, -0.8995833]},
                {"label": "thumb_04_right", "location": [0.07545345, 0.4856161, 0.0985662043], "orientation": [0.164432436, 0.265874773, 0.304997116, -0.8995833]},
                {"label": "upper_arm_right", "location": [0.143508673, 0.692019, -0.00195142627], "orientation": [0.161028028, -0.187570229, -0.132727951, 0.959828734]},
                {"label": "neck", "location": [0.031039834, 0.791987062, -0.06050851], "orientation": [0.0973199755, -0.09305143, 0.0139985494, 0.9907949]},
                {"label": "head", "location": [0.0220816135, 0.839961, -0.0209268127], "orientation": [-0.0121154524, 0.0030750595, -0.00578326825, 0.9999054]},
                {"label": "accessories_radix_nose", "location": [0.0243070759, 0.8812047, 0.0540360473], "orientation": [-0.0121154524, 0.0030750595, -0.00578326825, 0.9999054]},
                {"label": "eye_left", "location": [-0.000728130341, 0.8804352, 0.033081267], "orientation": [-0.0121154524, 0.0030750595, -0.00578326825, 0.9999054]},
                {"label": "eye_right", "location": [0.0482614972, 0.879857063, 0.03246386], "orientation": [-0.0121154524, 0.0030750595, -0.00578326825, 0.9999054]},
                {"label": "head_end", "location": [0.0239954218, 0.972520947, -0.01838562], "orientation": [-0.0121154524, 0.0030750595, -0.00578326825, 0.9999054]},
                {"label": "jaw", "location": [0.0225412473, 0.840232253, 6.735325E-06], "orientation": [-0.009623885, 0.003060665, -0.00579092558, 0.9999326]},
                {"label": "jaw_end", "location": [0.0225136243, 0.8089113, 0.04510912], "orientation": [-0.009623885, 0.003060665, -0.00579092558, 0.9999326]},
                {"label": "ear_left", "location": [-0.0422317274, 0.8757417, -0.02477517], "orientation": [-0.0121154524, 0.0030750595, -0.00578326825, 0.9999054]},
                {"label": "ear_right", "location": [0.0882902443, 0.8740893, -0.02649071], "orientation": [-0.0121154524, 0.0030750595, -0.00578326825, 0.9999054]},
                {"label": "nose", "location": [0.0240627043, 0.858897, 0.0669647157], "orientation": [-0.0121154524, 0.0030750595, -0.00578326825, 0.9999054]},
                {"label": "hip_left", "location": [-0.08443155, 0.5013434, -0.0025524348], "orientation": [-0.04940117, -0.03463017, -0.0623554476, 0.996228933]},
                {"label": "hip_right", "location": [0.0530355349, 0.484558344, 0.007887299], "orientation": [-0.04940117, -0.03463017, -0.0623554476, 0.996228933]}
            ]
        }
    ]
}

# 키포인트 위치와 방향
keypoints = data['keypoints'][0]['keypoints']

# 시각화
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

for keypoint in keypoints:
    label = keypoint['label']
    location = keypoint['location']
    orientation = keypoint['orientation']
    
    # 위치 표시
    ax.scatter(location[0], location[1], location[2], label=label)
    
    # 방향 표시 (간단하게 벡터의 시작점과 끝점으로 표시)
    orientation_vector = np.array(orientation[1:4])
    orientation_vector /= np.linalg.norm(orientation_vector)  # 단위 벡터로 정규화
    end_point = np.array(location) + orientation_vector * 0.1  # 방향 벡터를 적당히 길게 표시

    ax.plot([location[0], end_point[0]], [location[1], end_point[1]], [location[2], end_point[2]], color='r')

# 축 레이블
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')

# 범례를 플롯 밖으로 빼기
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

# 플롯 회전
ax.view_init(elev=20., azim=30)

plt.show()


In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
%matplotlib widget

labels=['Neck',
 'LeftArm',
 'LeftForeArm',
 'LeftHand',
 'RightArm',
 'RightForeArm',
 'RightHand',
 'LeftUpLeg',
 'LeftLeg',
 'LeftFoot',
 'LeftToeBase',
 'RightUpLeg',
 'RightLeg',
 'RightFoot',
 'RightToeBase']
# Provided array
test_points = np.array([[[-0.198224,  0.684328,  0.410513],
        [-0.140298,  0.541706,  0.359127],
        [ 0.129603,  0.351534,  0.330367],
        [ 0.352242,  0.319926,  0.417656],
        [-0.123035,  0.809991,  0.501271],
        [ 0.175541,  0.84528 ,  0.64072 ],
        [ 0.273193,  0.62785 ,  0.6778  ],
        [ 0.39953 ,  0.459665,  0.350853],
        [ 0.739233,  0.384324,  0.354854],
        [ 1.15107 ,  0.381675,  0.259092],
        [ 1.25324 ,  0.212305,  0.377694],
        [ 0.50183 ,  0.582197,  0.449925],
        [ 0.849722,  0.585595,  0.457034],
        [ 1.26741 ,  0.580871,  0.391463],
        [ 1.29868 ,  0.43066 ,  0.563309]]])

train_points = np.array([[[ 0.0196759 ,  0.224248  ,  0.192572  ],
        [ 0.204406  ,  0.225935  ,  0.238576  ],
        [ 0.368048  ,  0.222853  ,  0.50668   ],
        [ 0.406105  ,  0.130576  ,  0.759071  ],
        [-0.112816  ,  0.309666  ,  0.280817  ],
        [-0.147218  ,  0.421396  ,  0.570454  ],
        [-0.188246  ,  0.399867  ,  0.840545  ],
        [ 0.197175  ,  0.212092  ,  0.799738  ],
        [ 0.2972    ,  0.323141  ,  1.21565   ],
        [ 0.425012  ,  0.576068  ,  1.48924   ],
        [ 0.417159  ,  0.492314  ,  1.69069   ],
        [ 0.00405054,  0.252499  ,  0.814401  ],
        [-0.00543073,  0.241536  ,  1.27221   ],
        [ 0.0229774 ,  0.254687  ,  1.66553   ],
        [-0.0814027 ,  0.0903339 ,  1.74398   ]]])

# Extracting x, y, z coordinates
x = test_points[0, :, 0]
y = test_points[0, :, 1]
z = test_points[0, :, 2]

# Creating a 3D scatter plot
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(x, y, z)
ax.set_aspect('equal')

for i, label in enumerate(labels):
    ax.text(x[i], y[i], z[i], i)
# Labeling axes
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')

plt.show()

In [ ]:
## mo2cap2

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
%matplotlib widget
# Data
data = {
  "Neck": {
    "2d": [
      163.89647565048074,
      195.41000729373204
    ],
    "3d": [
      -3.51819,
      214.689,
      187.068
    ]
  },
  "LeftArm": {
    "2d": [
      198.37923041803808,
      177.1513546079833
    ],
    "3d": [
      145.56,
      194.231,
      256.129
    ]
  },
  "LeftForeArm": {
    "2d": [
      189.69791580085092,
      138.87781612178674
    ],
    "3d": [
      177.191,
      45.5225,
      502.309
    ]
  },
  "LeftHand": {
    "2d": [
      181.1005433790028,
      117.31245361631042
    ],
    "3d": [
      153.995,
      -144.741,
      675.459
    ]
  },
  "RightArm": {
    "2d": [
      124.20770468047516,
      175.41970080642406
    ],
    "3d": [
      -162.985,
      171.736,
      228.196
    ]
  },
  "RightForeArm": {
    "2d": [
      134.5621486845893,
      132.63820719349056
    ],
    "3d": [
      -204.766,
      0.838159,
      465.427
    ]
  },
  "RightHand": {
    "2d": [
      138.602755528115,
      106.63045288584534
    ],
    "3d": [
      -230.315,
      -226.457,
      586.356
    ]
  },
  "LeftUpLeg": {
    "2d": [
      169.261475043956,
      125.22851671904107
    ],
    "3d": [
      39.9429,
      -67.1389,
      670.816
    ]
  },
  "LeftLeg": {
    "2d": [
      161.27419497337283,
      111.04015340290918
    ],
    "3d": [
      -52.5097,
      -308.674,
      1020.89
    ]
  },
  "LeftFoot": {
    "2d": [
      157.56497052770573,
      109.86172240055298
    ],
    "3d": [
      -142.292,
      -437.807,
      1364.67
    ]
  },
  "LeftToeBase": {
    "2d": [
      164.35702826089963,
      102.66058351048164
    ],
    "3d": [
      -11.3093,
      -592.161,
      1371.76
    ]
  },
  "RightUpLeg": {
    "2d": [
      150.17050793589772,
      121.99122384683068
    ],
    "3d": [
      -128.215,
      -91.4276,
      622.687
    ]
  },
  "RightLeg": {
    "2d": [
      154.61545105690095,
      97.05269452326738
    ],
    "3d": [
      -135.179,
      -464.87,
      878.515
    ]
  },
  "RightFoot": {
    "2d": [
      156.5554408188637,
      91.66890268712993
    ],
    "3d": [
      -148.122,
      -722.675,
      1155.3
    ]
  },
  "RightToeBase": {
    "2d": [
      156.24557118233932,
      81.08717568855641
    ],
    "3d": [
      -151.866,
      -899.6,
      1062.12
    ]
  }
}
# Extracting the 3D coordinates
x = [data[key]["3d"][0] for key in data]
y = [data[key]["3d"][1] for key in data]
z = [data[key]["3d"][2] for key in data]
labels = [key for key in data]

# Plotting the 3D scatter plot
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# Scatter plot
ax.scatter(x, y, z)
ax.set_aspect('equal')
# Annotating the points
for i, label in enumerate(labels):
    ax.text(x[i], y[i], z[i], label)

# Setting labels
ax.set_xlabel('X Coordinate')
ax.set_ylabel('Y Coordinate')
ax.set_zlabel('Z Coordinate')

# Showing the plot
plt.show()


In [ ]:
labels

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import json


# Load the image
image_path = r'F:\extracted_mo2cap2_dataset\TrainSet\mo2cap2_chunk_0001\rgba\mo2cap2_chunk_0001_000014.png'
img = Image.open(image_path)
# Coordinates to plot
json_path = r'F:\extracted_mo2cap2_dataset\TrainSet\mo2cap2_chunk_0001\json\mo2cap2_chunk_0001_000014.json'
with open(json_path) as f:
	json_data = json.load(f)

coordinates = json_data
# Plot the image
plt.figure(figsize=(5, 5))
plt.imshow(img)

x = np.array([coordinates[key]["2d"][0] for key in coordinates]) - 33
y = np.array([coordinates[key]["2d"][1] for key in coordinates])

labels = [key for key in json_data]
for i, label in enumerate(labels):
    plt.text(x[i], y[i], label)
    
plt.scatter(x, y)

# Display the plot

plt.show()
print(json_data)